In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, when, sum as _sum,array, lit, col, concat_ws, when, split, size, array_remove

In [ ]:
spark = SparkSession.builder.appName("AITopic_BinarySearch").getOrCreate()

26/04/28 10:50:39 INFO SparkEnv: Registering MapOutputTracker
26/04/28 10:50:39 INFO SparkEnv: Registering BlockManagerMaster
26/04/28 10:50:39 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/28 10:50:39 INFO SparkEnv: Registering OutputCommitCoordinator


In [ ]:
df_posts = spark.read.parquet('gs://reddit-ai-2/process_data/Post_Single_or_Non_Tag/post_single_tag_or_non.parquet/part*')

df_posts.cache()

df_comments = spark.read.parquet('gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_single_tag_or_non.parquet/part*')

df_comments.cache()

DataFrame[body: string, created_utc: string, is_submitter: boolean, author: string, removal_reason: string, subreddit: string, score: bigint, ups: bigint, AI_Name: string]

In [ ]:
df_posts.printSchema()

print(df_posts.count())

df_posts.show(3)

root
 |-- content_categories: string (nullable = true)
 |-- created_utc: string (nullable = true)
 |-- author: string (nullable = true)
 |-- is_crosspostable: boolean (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- num_crossposts: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- title: string (nullable = true)
 |-- upvote_ratio: double (nullable = true)
 |-- ups: long (nullable = true)
 |-- downs: long (nullable = true)
 |-- view_count: string (nullable = true)
 |-- AI_Name: string (nullable = true)



255607
+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+------------------+---+-----+----------+-------+
|content_categories|        created_utc|            author|is_crosspostable|num_comments|num_crossposts|            selftext|subreddit|               title|      upvote_ratio|ups|downs|view_count|AI_Name|
+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+------------------+---+-----+----------+-------+
|              NULL|2025-12-07 15:50:47|        LancerNerd|            true|           1|             0|                    |  ChatGPT|here is my chatgp...|0.7200000286102295|  3|    0|      NULL|ChatGPT|
|              NULL|2025-12-07 15:52:25|Civil-Manager-5178|            true|          53|             0|It was doing so w...|  ChatGPT|  what just happened|0.860000014305114

In [ ]:
df_comments.printSchema()

print(df_comments.count())

df_comments.show(3)

root
 |-- body: string (nullable = true)
 |-- created_utc: string (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- removal_reason: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- score: long (nullable = true)
 |-- ups: long (nullable = true)
 |-- AI_Name: string (nullable = true)



3739964


+--------------------+-------------------+------------+-------------+--------------+---------+-----+---+-------+
|                body|        created_utc|is_submitter|       author|removal_reason|subreddit|score|ups|AI_Name|
+--------------------+-------------------+------------+-------------+--------------+---------+-----+---+-------+
|                \n\n|2026-02-11 07:50:13|       false|UnWindMachine|          NULL|  ChatGPT|    8|  8|ChatGPT|
|so we’re just giv...|2026-02-11 07:50:17|       false|WhipItWhippet|          NULL|  ChatGPT|    1|  1|ChatGPT|
|this sounds inter...|2026-02-11 07:50:21|       false|     Pijlpunt|          NULL|  ChatGPT|    3|  3|ChatGPT|
+--------------------+-------------------+------------+-------------+--------------+---------+-----+---+-------+
only showing top 3 rows



In [ ]:
topic_patterns = {
    # -------------------------
    # Use Cases
    # -------------------------
    "usecase_code": r"(?i)\b(?:code|cod(?:e|ing|er|ers)|dev(?:eloper|s)?|program(?:ming|mer|mers)?|script(?:ing)?|bug(?:gy)?|debug(?:ging)?|fix(?:ing|ed|es)?|patch(?:ed|ing|es)?|deploy(?:ment|ing|ed)?|compile(?:r|s|d|ing)?|build(?:ing|s|ed)?|syntax|repo(?:sitory)?|git|github|api(?:s)?|endpoint(?:s)?|algorithm(?:s)?|model(?:s)?|pipeline(?:s)?|backend|frontend|fullstack|sql|quer(?:y|ies|ying)|exception(?:s)?|error(?:s)?|crash(?:ed|ing|es)?|glitch(?:es|y)?|fail(?:ed|ing|s|ure)?|stacktrace|traceback|segfault|not\s+working|doesn['’]?t\s+work|won['’]?t\s+work|can['’]?t\s+run|failed\s+to\s+run|build\s+failed|runtime\s+error|execution\s+failed|keeps?\s+crashing|throws?\s+error|won['’]?t\s+start|just\s+stops?|keeps?\s+stopping|nothing\s+happens)\b",
    "usecase_write": r"(?i)\b(?:writ(?:e|ing|er|ers|ten)|email(?:s)?|draft(?:ing|ed|s)?|translat(?:e|ion|ing|ed)?|article(?:s)?|essay(?:s)?|blog(?:s|ging)?|paragraph(?:s)?|copywrit(?:e|ing|er|ers)|proofread(?:ing|er|s)?|grammar|rewrit(?:e|ing|ten)?|summari[sz](?:e|ing|ed|ation)?|text(?:s)?|doc(?:ument|s|umentation)?|report(?:ing|ed|s)?|content|caption(?:s)?|scriptwrit(?:e|ing|er)?|newsletter(?:s)?|post(?:ing|s)?|thread(?:s)?)\b",
    "usecase_data": r"(?i)\b(?:data|dataset(?:s)?|dataframe(?:s)?|excel|spreadsheet(?:s)?|csv|tsv|table(?:s)?|analytic(?:s|al)|analys(?:is|es|ing)|stat(?:s|istic(?:s|al)?)|numerical|quantitative|metric(?:s)?|kpi(?:s)?|dashboard(?:s)?|visuali[sz](?:e|ation|ing|ed)|chart(?:s)?|graph(?:s)?|plot(?:s)?|histogram(?:s)?|quer(?:y|ies)|aggregation(?:s)?|regression|classification|clustering)\b",
    "usecase_creative": r"(?i)\b(?:image(?:s)?|img(?:s)?|art(?:work)?|video(?:s)?|vid(?:s)?|design(?:ing|ed|s)?|photo(?:s)?|pic(?:s|ture(?:s)?)|graphic(?:s)?|animation(?:s)?|render(?:ing|ed|s)?|draw(?:ing|n|s)?|sketch(?:es)?|audio|sound(?:s)?|music|song(?:s)?|logo(?:s)?|illustration(?:s)?|visual(?:s)?|cinematic|edit(?:ing|ed|s)?|thumbnail(?:s)?|poster(?:s)?)\b",
    "usecase_research": r"(?i)\b(?:research(?:ing|er|ers)?|search(?:ing|es|ed)?|googl(?:e|ing|ed)|find(?:ing|s)?|explor(?:e|ing|ed)|stud(?:y|ies|ying)|info(?:rmation)?|knowledge|fact(?:s)?|explain(?:ing|ed|s|ation)?|paper(?:s)?|journal(?:s)?|source(?:s)?|cit(?:e|ation|ing|ed)|review(?:ing|ed|s)?|survey(?:s)?|accur(?:acy|ate|ately)|bias(?:ed)?|misinformation|hallucinat(?:ion|e|ing|ed)|incorrect|truth|validat(?:e|ion|ing|ed)|evidence|theory|theoretical|makes?\s+no\s+sense|doesn['’]?t\s+make\s+sense|weird\s+output|strange\s+result|random\s+output|inaccurate|factually\s+wrong|why\s+is\s+it\s+(?:doing|like\s+this)|what\s+is\s+this|what\s+even\s+is\s+this|feels?\s+off|seems?\s+off|doesn['’]?t\s+look\s+right)\b",
    # -------------------------
    # Product
    # -------------------------
    "prod_affordable": r"(?i)\b(?:price(?:s|y)?|cost(?:s|ly)?|pricing|plan(?:s)?|subscription(?:s)?|sub(?:s)?|premium|plus|pro|tier(?:s)?|fee(?:s)?|billing|invoice(?:s)?|purchas(?:e|ing|ed|es)|buy(?:ing|s)?|value|worth|expens(?:e|ive|ively)|cheap(?:er|est)?|afford(?:able|ability)|discount(?:s|ed|ing)?|budget(?:s)?|overpric(?:ed|ing)|underpric(?:ed|ing)|ripoff|paywall|free(?:tier)?|trial(?:s)?)\b",
    "prod_usability": r"(?i)\b(?:interface(?:s)?|ui|ux|navigat(?:e|ion|ing|ed)|user[-\s]?friendly|intuit(?:ive|ively)|accessib(?:le|ility)|layout(?:s)?|workflow(?:s)?|usability|complex(?:ity)?|simple|simplicity|confus(?:ing|ed|ion)|clunk(?:y|iness)|awkward(?:ly)?|friction|interaction(?:s)?|experience(?:s)?|onboarding|learnability|readability|hard\s+to\s+use|difficult\s+to\s+use|confusing\s+to\s+use|not\s+intuitive|poor\s+ux|bad\s+ui|pain\s+to\s+use)\b",
    "prod_performance": r"(?i)\b(?:performance|performant|speed|fast(?:er|est)?|slow(?:er|est)?|lag(?:gy|ging|ged)?|latenc(?:y|ies)|responsiv(?:e|eness)|delay(?:s|ed|ing)?|timeout(?:s)?|freez(?:e|ing|es|ed)|hang(?:ing|s)?|stuck|snapp(?:y|ier)?|smooth(?:ly)?|efficient(?:ly)?|efficiency|optimi[sz](?:e|ation|ing|ed)|throughput|scalab(?:le|ility)|fps|frame(?:rate)?|loading|buffer(?:ing)?|can['’]?t\s+use|unable\s+to\s+use|won['’]?t\s+load|not\s+loading|takes?\s+forever|super\s+slow|so\s+slow|lags?\s+a\s+lot|very\s+laggy|extremely\s+slow|unresponsive|stops?\s+responding|not\s+responding|freezes?\s+up|can['’]?t\s+do\s+anything|nothing\s+happens)\b",
    # -------------------------
    # User Behavior
    # -------------------------
    "user_churn": r"(?i)\b(?:cancel(?:led|ling|s)?|unsubscribe(?:d|s)?|quit(?:ting|s)?|leav(?:e|ing|es)|stop(?:ped|ping)?\s+using|switch(?:ing|es)?\s+to|mov(?:e|ed|ing)\s+to|migrat(?:e|ing|ed|ion)|abandon(?:ed|ing)?|uninstall(?:ed|ing)?|downgrad(?:e|ing|ed)|terminat(?:e|ing|ed|ion)|discontinu(?:e|ed|ing)|refund(?:ed|s|ing)?|not\s+worth|no\s+longer\s+using|gave\s+up|dropped\s+it|i['’]?m\s+done|won['’]?t\s+use\s+again)\b",
    # -------------------------
    # Sentiment
    # -------------------------
    "sentiment_positive": r"(?i)\b(?:good|great|awesome|amazing|excellent|nice|solid|love(?:d|s|ing)?|like(?:d|s|ing)?|useful|helpful|impress(?:ive|ively)|fantastic|perfect|cool|help|helps|smooth|fast|clean|neat|well\s?done)\b",
    "sentiment_negative": r"(?i)\b(?:bad|terrible|awful|horrible|hate(?:d|s|ing)?|dislike(?:d|s|ing)?|useless|annoy(?:ing|ed|s)|frustrat(?:ing|ed|ion)|disappoint(?:ing|ed|ment)|worst|trash|garbage|suck(?:s|ed|ing)?|cooked|broken|laggy|slow|messy|cluttered|bugged|shit|bullshit|fuck|noob|this\s+ain['’]?t\s+it|nah|bruh|wtf|what\s+the\s+fuck|lol\s+this\s+sucks|i['’]?m\s+confused|this\s+is\s+confusing|i\s+don['’]?t\s+get\s+it|this\s+is\s+odd|a\s+bit\s+off|something['’]?\s+s\s+off|stupid)\b"
}

In [ ]:
# Function for Binary Search and Result Summary
def extract_and_summarize_topics(df, text_column):
    # Define the column to use for topic detection.
    if text_column == "selftext":
        df_processed = df.withColumn(
            "text_for_topic_detection",
            when(
                (lower(col(text_column)) == "[removed]") |
                (lower(col(text_column)) == "[deleted]") |
                (lower(col(text_column)) == "[removed by moderator]") |
                (col(text_column).isNull()),
                col("title")
            ).otherwise(col(text_column))
        )
    else:
        df_processed = df.withColumn("text_for_topic_detection", col(text_column))

    # Filter out null or empty values from the new 'text_for_topic_detection' column
    #df_clean = df_processed.filter(col("text_for_topic_detection").isNotNull() & (col("text_for_topic_detection") != ""))
    df_clean = df_processed

    # Loop through each regex condition, assign 1 if found, 0 if not found
    for topic_name, pattern in topic_patterns.items():
        df_clean = df_clean.withColumn(
            f"flag_{topic_name}",
            when(lower(col("text_for_topic_detection")).rlike(pattern), 1).otherwise(0)
        )

    # Summarize the total count of posts/comments mentioning each topic
    agg_expressions = [
        _sum(col(f"flag_{topic_name}")).alias(f"Total_{topic_name}")
        for topic_name in topic_patterns.keys()
    ]

    df_summary = df_clean.agg(*agg_expressions)
    return df_clean, df_summary

In [ ]:
df_posts_flagged, df_posts_summary = extract_and_summarize_topics(df_posts, "selftext")
df_posts_flagged.cache() # Cache for reuse to optimize subsequent operations
df_comments_flagged, df_comments_summary = extract_and_summarize_topics(df_comments, "body")
df_comments_flagged.cache() # Cache for reuse to optimize subsequent operations

26/04/28 10:47:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[body: string, created_utc: string, is_submitter: boolean, author: string, removal_reason: string, subreddit: string, score: bigint, ups: bigint, AI_Name: string, text_for_topic_detection: string, flag_usecase_code: int, flag_usecase_write: int, flag_usecase_data: int, flag_usecase_creative: int, flag_usecase_research: int, flag_prod_affordable: int, flag_prod_usability: int, flag_prod_performance: int, flag_user_churn: int, flag_sentiment_positive: int, flag_sentiment_negative: int]

In [ ]:
# Get all flag columns
flag_columns = [f"flag_{topic}" for topic in topic_patterns.keys()]

# Function to create a list of detected topics for a given DataFrame
def add_detected_topics_column(df_flagged, topic_prefix="flag_"):
    # Create a list of expressions, each yielding topic name if flagged, else an empty string
    topic_conditional_expressions = []
    for topic_name in topic_patterns.keys():
        flag_col_name = f"{topic_prefix}{topic_name}"
        # If flag is 1, return topic_name, otherwise return an empty string to be filtered out
        topic_conditional_expressions.append(when(col(flag_col_name) == 1, lit(topic_name)).otherwise(lit("")))

    # Create an array that contains topic names and potentially empty strings
    df_with_topics = df_flagged.withColumn(
        "temp_detected_topics_array_with_empties", array(*topic_conditional_expressions)
    )

    # Remove empty strings from the array.
    df_with_topics = df_with_topics.withColumn(
        "detected_topics", array_remove(col("temp_detected_topics_array_with_empties"),"")
    ).drop("temp_detected_topics_array_with_empties")

    return df_with_topics

# Apply to posts DataFrame
df_posts_with_topics = add_detected_topics_column(df_posts_flagged)
print("--- Schema of df_posts_with_topics ---")
df_posts_with_topics.printSchema()
print("--- Sample of df_posts_with_topics (only 'selftext' and 'detected_topics' columns) ---")
df_posts_with_topics.select("detected_topics","text_for_topic_detection").limit(5).show(truncate=False)

# Apply to comments DataFrame
df_comments_with_topics = add_detected_topics_column(df_comments_flagged)
print("\n--- Schema of df_comments_with_topics ---")
df_comments_with_topics.printSchema()
print("--- Sample of df_comments_with_topics (only 'body' and 'detected_topics' columns) ---")
df_comments_with_topics.select("detected_topics","body").limit(5).show(truncate=False)

--- Schema of df_posts_with_topics ---
root
 |-- content_categories: string (nullable = true)
 |-- created_utc: string (nullable = true)
 |-- author: string (nullable = true)
 |-- is_crosspostable: boolean (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- num_crossposts: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- title: string (nullable = true)
 |-- upvote_ratio: double (nullable = true)
 |-- ups: long (nullable = true)
 |-- downs: long (nullable = true)
 |-- view_count: string (nullable = true)
 |-- AI_Name: string (nullable = true)
 |-- text_for_topic_detection: string (nullable = true)
 |-- flag_usecase_code: integer (nullable = false)
 |-- flag_usecase_write: integer (nullable = false)
 |-- flag_usecase_data: integer (nullable = false)
 |-- flag_usecase_creative: integer (nullable = false)
 |-- flag_usecase_research: integer (nullable = false)
 |-- flag_prod_affordable: integer (nullable = false)
 |-- 

+---------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|detected_topics                                    |text_for_topic_detection                                                                                                                                                                                                                                                                                                                               |
+---------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------

+------------------+-----------------------------------------------------------------------------------------------------+
|detected_topics   |body                                                                                                 |
+------------------+-----------------------------------------------------------------------------------------------------+
|[]                |\n\n                                                                                                 |
|[]                |so we’re just giving ais cocaine now huh                                                             |
|[usecase_creative]|this sounds interesting. i would be interested in giving it a try, please dm me when you have a beta.|
|[prod_affordable] |not true. chatgpt itself mentions india, turkey and argentina as having the cheapest pricing plans.  |
|[]                |awsome                                                                                               |
+---------------

In [ ]:
# Calculate total counts
total_posts = df_posts.count()
total_comments = df_comments.count()

print(f"\n--- Topic Proportions in Posts (out of {total_posts} rows) ---")
# Use Spark to calculate proportions without converting to Pandas DataFrame
posts_summary_row = df_posts_summary.collect()[0]
for col_name in df_posts_summary.columns:
    topic_name = col_name.replace('Total_', '')
    count = posts_summary_row[col_name]
    proportion = count / total_posts
    print(f"  {topic_name}: {proportion:.4f} ({count} detections)")

# Calculate and display posts with no detected topics
posts_no_topic_detected = df_posts_with_topics.filter(size(col("detected_topics")) == 0).count()
proportion_posts_no_topic = posts_no_topic_detected / total_posts
print(f"  No topics detected: {proportion_posts_no_topic:.4f} ({posts_no_topic_detected} detections)")

print(f"\n--- Topic Proportions in Comments (out of {total_comments} rows) ---")
# Use Spark to calculate proportions without converting to Pandas DataFrame
comments_summary_row = df_comments_summary.collect()[0]
for col_name in df_comments_summary.columns:
    topic_name = col_name.replace('Total_', '')
    count = comments_summary_row[col_name]
    proportion = count / total_comments
    print(f"  {topic_name}: {proportion:.4f} ({count} detections)")

# Calculate and display comments with no detected topics
comments_no_topic_detected = df_comments_with_topics.filter(size(col("detected_topics")) == 0).count()
proportion_comments_no_topic = comments_no_topic_detected / total_comments
print(f"  No topics detected: {proportion_comments_no_topic:.4f} ({comments_no_topic_detected} detections)")


--- Topic Proportions in Posts (out of 255607 rows) ---
  usecase_code: 0.2239 (57240 detections)
  usecase_write: 0.1545 (39481 detections)
  usecase_data: 0.0585 (14960 detections)
  usecase_creative: 0.1472 (37636 detections)
  usecase_research: 0.1818 (46474 detections)
  prod_affordable: 0.1449 (37030 detections)
  prod_usability: 0.0930 (23760 detections)
  prod_performance: 0.0675 (17260 detections)
  user_churn: 0.0338 (8627 detections)
  sentiment_positive: 0.2537 (64851 detections)
  sentiment_negative: 0.0923 (23605 detections)
  No topics detected: 0.4798 (122637 detections)

--- Topic Proportions in Comments (out of 3739964 rows) ---


  usecase_code: 0.0930 (347711 detections)
  usecase_write: 0.0527 (197232 detections)
  usecase_data: 0.0132 (49475 detections)
  usecase_creative: 0.0537 (200816 detections)
  usecase_research: 0.0631 (235889 detections)
  prod_affordable: 0.0513 (191831 detections)
  prod_usability: 0.0160 (59748 detections)
  prod_performance: 0.0138 (51605 detections)
  user_churn: 0.0099 (37103 detections)
  sentiment_positive: 0.1597 (597160 detections)
  sentiment_negative: 0.0612 (228719 detections)


  No topics detected: 0.5687 (2126894 detections)


In [ ]:
output_path_posts = "gs://reddit-ai-2/process_data/Post_Single_or_Non_Tag/post_with_topic.parquet"
output_path_comments = 'gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_with_topic.parquet'

# Define columns to drop
columns_to_drop = ["text_for_topic_detection"] + [f"flag_{topic}" for topic in topic_patterns.keys()]

In [ ]:
print(df_comments_with_topics.count())

3739964


In [ ]:
# Drop specified columns from df_posts_with_topics before saving
df_posts_to_save = df_posts_with_topics.drop(*columns_to_drop)

# Save df_posts_to_save to Parquet
df_posts_to_save.coalesce(1).write.mode("overwrite").parquet(output_path_posts)
print(f"df_posts_with_topics saved to {output_path_posts}")

# Drop specified columns from df_comments_with_topics before saving
df_comments_to_save = df_comments_with_topics.drop(*columns_to_drop)

# Save df_comments_to_save to Parquet
df_comments_to_save.coalesce(1).write.mode("overwrite").parquet(output_path_comments)
print(f"df_comments_with_topics saved to {output_path_comments}")

df_posts_with_topics saved to gs://reddit-ai-2/process_data/Post_Single_or_Non_Tag/post_with_topic.parquet


df_comments_with_topics saved to gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_with_topic.parquet


In [ ]:
print(df_posts_to_save.count())
print(df_comments_to_save.count())

255607
3739964


In [ ]:
#output_path_posts_sample = "gs://reddit-ai-2/process_data/Sentiment_Train_DataSet/sample_posts_for_deploy.parquet"
#output_path_comments_sample = "gs://reddit-ai-2/process_data/Sentiment_Train_DataSet/sample_comments_for_deploy.parquet"

#df_posts_to_save.limit(1000).coalesce(1).write.mode("overwrite").parquet(output_path_posts_sample)
#df_comments_to_save.limit(5000).coalesce(1).write.mode("overwrite").parquet(output_path_comments_sample)

In [ ]:
spark.stop()
print("SparkSession stopped.")

org.apache.spark.SparkException: Could not find CoarseGrainedScheduler.
	at org.apache.spark.rpc.netty.Dispatcher.postMessage(Dispatcher.scala:178)
	at org.apache.spark.rpc.netty.Dispatcher.postOneWayMessage(Dispatcher.scala:150)
	at org.apache.spark.rpc.netty.NettyRpcEnv.send(NettyRpcEnv.scala:193)
	at org.apache.spark.rpc.netty.NettyRpcEndpointRef.send(NettyRpcEnv.scala:563)
	at org.apache.spark.scheduler.cluster.YarnSchedulerBackend$YarnSchedulerEndpoint.$anonfun$org$apache$spark$scheduler$cluster$YarnSchedulerBackend$$handleExecutorDisconnectedFromDriver$3(YarnSchedulerBackend.scala:319)
	at org.apache.spark.scheduler.cluster.YarnSchedulerBackend$YarnSchedulerEndpoint.$anonfun$org$apache$spark$scheduler$cluster$YarnSchedulerBackend$$handleExecutorDisconnectedFromDriver$3$adapted(YarnSchedulerBackend.scala:319)
	at scala.util.Success.foreach(Try.scala:253)
	at scala.concurrent.Future.$anonfun$foreach$1$adapted(Future.scala:229)
	at scala.concurrent.impl.CallbackRunnable.run(Promise.

SparkSession stopped.
